In [7]:
from langchain.agents import create_agent
from langchain_text_splitters import MarkdownHeaderTextSplitter

with open("resource/attention-is-all-you-need-Paper.md", "r") as f:
    doc = f.read()

headers_to_split_on = [
    ("#", "H1"),
    ("##", "H2"),
    ("###", "H3"),
    ("####", "H4"),
    ("#####", "H5"),
    ("######", "H6")
]

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False, # 是否去除标题
)

# 分割文档
documents = splitter.split_text(doc)

for i, document in enumerate(documents):
    document.id = f"document_{i}"

print(documents)

[Document(id='document_0', metadata={'H1': 'Attention Is All You Need'}, page_content='# Attention Is All You Need  \nAshish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin. 发表于 NIPS 2017。本文档根据原论文整理，便于 RAG / Markdown 标题切分测试。'), Document(id='document_1', metadata={'H1': 'Attention Is All You Need', 'H2': 'Abstract'}, page_content='## Abstract  \n主流序列转导模型基于复杂的循环或卷积神经网络，并包含编码器与解码器；表现最好的模型还会通过注意力机制连接编码器与解码器。本文提出一种全新的简单网络架构——Transformer，完全基于注意力机制，彻底摒弃循环与卷积。在两项机器翻译任务上的实验表明，该模型质量更优、更易并行，且训练时间显著更短。模型在 WMT 2014 英德翻译任务上达到 28.4 BLEU，比此前最佳结果（含集成）高出超过 2 BLEU；在 WMT 2014 英法翻译任务上，单模型达到新的 SOTA BLEU 分数 41.0，仅在 8 块 GPU 上训练 3.5 天，远低于文献中最佳模型的训练成本。'), Document(id='document_2', metadata={'H1': 'Attention Is All You Need', 'H2': '一、Introduction'}, page_content='## 一、Introduction  \n循环神经网络，尤其是 LSTM 与门控循环网络，已是序列建模与转导问题（如语言模型与机器翻译）的主流方法。循环模型通常沿输入输出序列的符号位置分解计算：将位置对齐到计算时间步，生成隐藏状态序列 ht，它是前一隐藏状态 ht−1 与位置 t 输入的函数。这种固有的序列性阻碍了训练样本内部的并行化；序列越长，受内存限制，跨样本 ba

## 准备知识库

In [8]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings

ollama_embedding = OllamaEmbeddings(
    model="qwen3-embedding:0.6b"
)
vector_store = InMemoryVectorStore(ollama_embedding)

vector_store.add_documents(documents)

['document_0',
 'document_1',
 'document_2',
 'document_3',
 'document_4',
 'document_5',
 'document_6',
 'document_7',
 'document_8',
 'document_9',
 'document_10',
 'document_11',
 'document_12',
 'document_13',
 'document_14',
 'document_15',
 'document_16',
 'document_17',
 'document_18',
 'document_19',
 'document_20',
 'document_21',
 'document_22',
 'document_23',
 'document_24',
 'document_25',
 'document_26',
 'document_27',
 'document_28',
 'document_29',
 'document_30',
 'document_31',
 'document_32',
 'document_33',
 'document_34',
 'document_35',
 'document_36',
 'document_37',
 'document_38',
 'document_39',
 'document_40',
 'document_41',
 'document_42',
 'document_43',
 'document_44',
 'document_45',
 'document_46',
 'document_47',
 'document_48',
 'document_49']

## 2-step模式

In [9]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain.chat_models import init_chat_model

@dynamic_prompt
def rag_prompt(req: ModelRequest):
    # 拿到用户的问题
    user_question = req.messages[-1].content
    print("user_question:", user_question)
    # 检索知识片段
    results = vector_store.similarity_search(user_question, k=2)
    print("results:", results)
    content = "\n".join([doc.page_content for doc in results])
    # 构建提示词
    prompt = f"""
    你是一个Transformer方面的专家，你根据上下文回答用户问题，不能自己编造答案。答案要简短，直接引用原文
    Context：{content}
    """
    return prompt

model = init_chat_model(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

agent = create_agent(
    model=model,
    middleware=[rag_prompt],
)

In [13]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Transformer的核心思想是什么？")]}
)
print("response:", response["messages"][-1].content)

user_question: Transformer的核心思想是什么？
results: [Document(id='document_3', metadata={'H1': 'Attention Is All You Need', 'H2': '一、Introduction', 'H3': '1.1 核心贡献'}, page_content='### 1.1 核心贡献  \nTransformer 是首个完全依赖自注意力计算输入与输出表示、且不使用序列对齐 RNN 或卷积的转导模型。'), Document(id='document_20', metadata={'H1': 'Attention Is All You Need', 'H2': '三、Model Architecture', 'H3': '3.2 Attention', 'H4': '3.2.2 Multi-Head Attention', 'H5': '超参数设定', 'H6': 'Multi-Head 的好处'}, page_content='###### Multi-Head 的好处  \nMulti-head attention 使模型能共同关注不同位置、不同表示子空间的信息；单头时平均会抑制这种能力。')]
response: Transformer 是首个完全依赖自注意力计算输入与输出表示、且不使用序列对齐 RNN 或卷积的转导模型。


## agentic Rag

In [16]:
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from dotenv import load_dotenv

load_dotenv()

model = init_chat_model(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

@tool
def retrieve_docs(query: str) -> str:
    """检索文档，返回2个最相关的文档"""
    results = vector_store.similarity_search(query, k=2)
    return "\n".join([doc.page_content for doc in results])

agent = create_agent(
    model=model,
    tools=[retrieve_docs],
    system_prompt="你是一个Transformer方面的专家，在回答Transformer相关问题时，先检索文档，然后根据文档回答问题，回答要简短"
)


In [17]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Transformer的核心思想是什么？")]}
)
for msg in response["messages"]:
    print(msg.pretty_print())

response = agent.invoke(
    {"messages": [HumanMessage(content="你好")]}
)
for msg in response["messages"]:
    print(msg.pretty_print())

================================ Human Message =================================

Transformer的核心思想是什么？
None
================================== Ai Message ==================================
Tool Calls:
  retrieve_docs (call_00_96765Buq7OYUSOlPXJlF7971)
 Call ID: call_00_96765Buq7OYUSOlPXJlF7971
  Args:
    query: Transformer 核心思想
None
================================= Tool Message =================================
Name: retrieve_docs

### 1.1 核心贡献  
Transformer 是首个完全依赖自注意力计算输入与输出表示、且不使用序列对齐 RNN 或卷积的转导模型。
## 七、Conclusion  
本文提出 Transformer：首个完全基于注意力的序列转导模型，用 multi-headed self-attention 取代 encoder-decoder 中常用的循环层。在翻译任务上，Transformer 可比基于循环或卷积的架构训练得显著更快；在 WMT 2014 英德与英法任务上均达到新 SOTA，英德任务上最佳模型甚至超过此前所有已报道的集成模型。  
作者计划将 Transformer 扩展到文本以外的输入输出模态，并研究局部/受限注意力以高效处理图像、音频、视频等大输入输出；使生成更少序列依赖也是研究目标之一。训练与评估代码见：https://github.com/tensorflow/tensor2tensor
None
================================== Ai Message ==================================

Transformer 的核心思想：**完全用自注意力（self-attention）来建模序列，彻底抛弃循环（RNN）和卷积